# Budget Allocation Analysis

Exploring household income, expense patterns, and budget adherence across Indian households.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/BudgetAllocation.csv')
print('Shape:', df.shape)
df.head(2)

In [ ]:
print('=== Data Types ===')
df.info()
print()
print('=== Summary Stats ===')
df.describe()
print()
print('=== Null Values ===')
print(df.isnull().sum())

## 1. Income Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='Income', bins=30, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Income Distribution')
axes[0].axvline(df['Income'].mean(), color='red', ls='--', label=f"Mean: {df['Income'].mean():.0f}")
axes[0].axvline(df['Income'].median(), color='green', ls='--', label=f"Median: {df['Income'].median():.0f}")
axes[0].legend()

sns.boxplot(data=df, x='Income', color='steelblue', ax=axes[1])
axes[1].set_title('Income Box Plot')

plt.tight_layout()
plt.show()

print(f"Income range: {df['Income'].min():.0f} - {df['Income'].max():.0f}")
print(f"Mean: {df['Income'].mean():.0f}, Median: {df['Income'].median():.0f}")

## 2. Average Spending Pattern

In [ ]:
expense_cols = ['HousingExpense', 'TransportationExpense', 'FoodExpense',
                'UtilitiesExpense', 'EntertainmentExpense', 'Savings']
budget_cols = ['HousingBudget', 'TransportationBudget', 'FoodBudget',
               'UtilitiesBudget', 'EntertainmentBudget', 'SavingsBudget']
labels = ['Housing', 'Transport', 'Food', 'Utilities', 'Entertainment', 'Savings']

avg_expenses = df[expense_cols].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

colors = sns.color_palette('Set2', n_colors=6)
wedges, texts, autotexts = axes[0].pie(
    avg_expenses.values, labels=labels, autopct='%1.1f%%',
    startangle=90, colors=colors, explode=[0.02]*6)
axes[0].set_title('Average Expense Breakdown')

bar_ax = axes[1]
bars = bar_ax.bar(labels, avg_expenses.values, color=colors, edgecolor='white')
bar_ax.set_title('Average Monthly Expense (Absolute)')
bar_ax.set_ylabel('Amount')
for bar, val in zip(bars, avg_expenses.values):
    bar_ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{val:.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 3. Budget vs Actual Spending (Percentage)

In [ ]:
# Calculate actual % of income for each expense
for col in expense_cols:
    df[f'{col}_pct'] = df[col] / df['Income'] * 100

actual_pct_cols = [f'{c}_pct' for c in expense_cols]
avg_actual_pct = df[actual_pct_cols].mean()
avg_budget_pct = df[budget_cols].mean()

comp_df = pd.DataFrame({
    'Category': labels,
    'Actual %': avg_actual_pct.values,
    'Budgeted %': avg_budget_pct.values
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

x = np.arange(len(labels))
w = 0.35
axes[0].bar(x - w/2, comp_df['Actual %'], w, label='Actual %', color='#4C72B0')
axes[0].bar(x + w/2, comp_df['Budgeted %'], w, label='Budgeted %', color='#DD8452')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=45)
axes[0].set_ylabel('Percentage of Income')
axes[0].set_title('Actual vs Budgeted Spending (% of Income)')
axes[0].legend()

comp_df['Difference'] = comp_df['Actual %'] - comp_df['Budgeted %']
bar_colors = ['#2ca02c' if v < 0 else '#d62728' for v in comp_df['Difference']]
bars = axes[1].bar(labels, comp_df['Difference'], color=bar_colors, edgecolor='white')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Difference (Actual % - Budgeted %)')
axes[1].set_ylabel('Percentage Points')
axes[1].tick_params(axis='x', rotation=45)
for bar, val in zip(bars, comp_df['Difference']):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + (0.2 if val >= 0 else -0.6),
                 f'{val:+.1f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

comp_df.round(2)

## 4. Savings Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='Savings', bins=30, kde=True, color='teal', ax=axes[0])
axes[0].set_title('Savings Distribution')
axes[0].axvline(df['Savings'].mean(), color='red', ls='--', label=f"Mean: {df['Savings'].mean():.0f}")
axes[0].axvline(df['Savings'].median(), color='green', ls='--', label=f"Median: {df['Savings'].median():.0f}")
axes[0].legend()

sns.scatterplot(data=df, x='Income', y='Savings', alpha=0.5, color='teal', ax=axes[1])
axes[1].set_title('Income vs Savings')
axes[1].set_xlabel('Income')
axes[1].set_ylabel('Savings')

from numpy.polynomial.polynomial import polyfit
x, y = df['Income'], df['Savings']
b, m = polyfit(x, y, 1)
axes[1].plot(x, b + m * x, color='red', lw=2, ls='--', label=f'Trend (slope={m:.2f})')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Savings rate (avg): {df['Savings'].mean() / df['Income'].mean() * 100:.1f}%")
print(f"Households with negative savings: {(df['Savings'] < 0).sum()}")

## 5. Income vs Expense Categories

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (col, label) in enumerate(zip(expense_cols, labels)):
    sns.scatterplot(data=df, x='Income', y=col, alpha=0.4, color='steelblue', ax=axes[i])
    axes[i].set_title(f'Income vs {label}')
    axes[i].set_xlabel('Income')
    axes[i].set_ylabel(label)
    b, m = polyfit(df['Income'], df[col], 1)
    axes[i].plot(df['Income'], b + m * df['Income'], color='red', lw=1.5, ls='--')

plt.tight_layout()
plt.show()

## 6. Budget Adherence Distribution

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (col, actual_col, label) in enumerate(zip(budget_cols, actual_pct_cols, labels)):
    diff = df[actual_col] - df[col]
    sns.histplot(diff, bins=30, kde=True, color='purple', ax=axes[i])
    axes[i].axvline(0, color='black', lw=1.5)
    axes[i].axvline(diff.mean(), color='red', ls='--', label=f"Mean: {diff.mean():+.2f}")
    axes[i].set_title(f'{label} — Actual vs Budget Difference')
    axes[i].set_xlabel('Percentage Points')
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. Correlation Heatmap

In [ ]:
all_cols = ['Income'] + expense_cols + ['Savings'] + budget_cols

plt.figure(figsize=(11, 9))
corr = df[all_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, square=True, linewidths=0.8,
            xticklabels=labels + ['Savings'] + [l+'%' for l in labels],
            yticklabels=labels + ['Savings'] + [l+'%' for l in labels])
plt.title('Correlation Matrix of Income, Expenses & Budgets')
plt.tight_layout()
plt.show()

## 8. Key Takeaways

- **Income skew**: Income distribution is right-skewed (mean ≈ 10K, max ≈ 17.7K). Most households cluster in the 8K–12K range.
- **Spending pattern**: Food accounts for the largest expense share (~32%), followed by Housing (~23%) and Transportation (~6%). Savings average ~11%.
- **Budget vs Actual**: Households consistently underspend on Housing (actual ~6 pp below budget) and overspend on Food (~5 pp above budget). Savings match budgeted targets closely.
- **Savings**: Savings rise with income, but the savings rate varies widely. Some households (very few) have negative savings.
- **Income sensitivity**: Food and Housing expenses increase with income (positive correlation), while Transportation and Entertainment show weaker income dependence.
- **Budget adherence variability**: Entertainment and Transportation budgets show the widest variance in adherence — some households under-budget, others overspend significantly.
- Overall, households follow their budgets reasonably well but systematically underestimate Food costs while over-allocating to Housing.